# Liver Tumor Segmentation — Stage 1 (Kaggle Notebook)

**Workflow:**
1. Add the MSD Task03 Liver dataset (search 'MSD Task03 Liver' under '+ Add Data') OR use your own uploaded version.
2. Add your code (either via GitHub clone or as a Kaggle Dataset).
3. Run the preprocessing cell **ONCE** (saves processed NIfTIs to /kaggle/working).
4. Click **Save Version → Save & Run All (Commit)**. Kaggle will run this notebook in the background (up to 12 hours).
5. The committed notebook's /kaggle/working becomes a Kaggle Dataset; you can attach it to future notebooks so preprocessing is never redone.

Make sure 'Accelerator' is set to **GPU P100** in the right sidebar before running.

In [ ]:
# Cell 1: Environment setup
import os, sys, time
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory / 1e9, 'GB')

In [ ]:
# Cell 2: Get the code
# OPTION A — GitHub clone (replace URL with your repo):
# !git clone https://github.com/<your-user>/<your-repo>.git /kaggle/working/liver_project
# %cd /kaggle/working/liver_project

# OPTION B — Code uploaded as Kaggle Dataset (e.g., named 'liver-project-code'):
# !cp -r /kaggle/input/liver-project-code/* /kaggle/working/
# %cd /kaggle/working

# Append src/ to path so 'from src.xxx import' works
sys.path.insert(0, '/kaggle/working')
!ls

In [ ]:
# Cell 3: Preprocessing (ONE-TIME)
# Run your Week 1 preprocessing modules on the raw MSD dataset.
# Adjust DATA_DIR to match the actual path of your added dataset.
DATA_DIR = '/kaggle/input/msd-task03-liver/Task03_Liver'  # <-- adjust
PROCESSED_DIR = '/kaggle/working/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Quick check: is data there?
import os
for d in [DATA_DIR, os.path.join(DATA_DIR, 'imagesTr'), os.path.join(DATA_DIR, 'labelsTr')]:
    print(d, 'exists' if os.path.exists(d) else 'MISSING')

# Call your preprocessing pipeline here. Example (adjust to your actual module names):
# from src.preprocess import process_all
# process_all(input_dir=DATA_DIR, output_dir=PROCESSED_DIR)

In [ ]:
# Cell 4: Build 5-fold split
!python -m src.cv_split --processed_dir $PROCESSED_DIR --out /kaggle/working/folds.json

In [ ]:
# Cell 5: Sanity test — UNet3D forward pass
from src.models.unet3d import UNet3D, count_parameters
model = UNet3D(in_channels=1, out_channels=1, base_features=32, levels=4).cuda()
print(f'Parameters: {count_parameters(model):,}')
x = torch.randn(2, 1, 96, 96, 96, device='cuda')
with torch.cuda.amp.autocast():
    y = model(x)
print('Output:', y.shape)
print('Peak VRAM:', torch.cuda.max_memory_allocated() / 1e9, 'GB')
del model, x, y; torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

In [ ]:
# Cell 6: Stage 1 training — Fold 0
# At batch_size=2 and 96^3 patches, expect ~3-4 sec/iter on P100.
# With 105 train * 4 samples_per_volume / batch_size 2 = 210 iter/epoch.
# So ~12-15 min/epoch. 50 epochs ~10-12 hours.
# Reduce epochs to 30 if you want a faster first run.

!python -m src.train_stage1 \
    --processed_dir $PROCESSED_DIR \
    --folds_json /kaggle/working/folds.json \
    --fold 0 \
    --out_dir /kaggle/working/runs/stage1_fold0 \
    --epochs 40 \
    --batch_size 2 \
    --samples_per_volume 4 \
    --base_features 32 \
    --val_every 5 \
    --num_workers 2

In [ ]:
# Cell 7: Plot training curves
import json
import matplotlib.pyplot as plt
with open('/kaggle/working/runs/stage1_fold0/log.json') as f:
    log = json.load(f)
epochs = [r['epoch'] for r in log]
losses = [r['train_loss'] for r in log]
val_dsc = [r.get('val_dsc_class_1') for r in log]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(epochs, losses); ax[0].set_title('train loss'); ax[0].grid(True)
ax[1].plot([e for e,d in zip(epochs, val_dsc) if d is not None],
           [d for d in val_dsc if d is not None], 'o-')
ax[1].set_title('val DSC (liver)'); ax[1].grid(True); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()